In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

from ISLP.models import (ModelSpec as MS, summarize , poly)

# Multilinear Regression

## Construct Toy Data

In [ ]:
n = 100
rng = np.random.default_rng(1234)

x1_ = rng.uniform(0, 10, n)
x2_ = rng.uniform(-5, 5, n)
y_ = 2 - 1.5 * x1_ + 3 * x2_ + rng.normal(size=x1_.shape)

toy_df = pd.DataFrame({'x1': x1_, 'x2': x2_, 'y': y_})
toy_df.head()


In [ ]:

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.scatter(toy_df['x1'], toy_df['x2'], toy_df['y'])
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_zlabel(r'$y$')

## Fit with statsmodels

### Manual Construction of Design Matrix

In [ ]:
X = toy_df[['x1', 'x2']]
X = sm.add_constant(X) # handles the intercept, beta0
X.head()

In [ ]:
model=sm.OLS(toy_df['y'], X)
results=model.fit()


In [ ]:
results.summary()

In [ ]:
summarize(results)

### Construction with Formula

In [ ]:

model=smf.ols('y~x1 + x2', data=toy_df)
results=model.fit()
summarize(results)

This the same as in the explicit design matrix construction

### Construction with ModelSpec
This uses conveneience functions, provided by the textbook authors.  Fundamentally, its built on `sklearn` and `statsmodels`

In [ ]:
design = MS(['x1','x2']) # design matrix specification 
design

In [ ]:
X = design.fit_transform(toy_df)
X.head()

In [ ]:
model = sm.OLS(toy_df['y'], X)
results = model.fit()
summarize(results)

Same as before.  Why bother?  
Suppoose we have new data.

In [ ]:
nnew = 20
rng = np.random.default_rng(468)

x1_ = rng.uniform(0, 10, nnew)
x2_ = rng.uniform(-5, 5, nnew)

# this avoids having to pad with the 1 for the intercept
new_df = pd.DataFrame({'x1': x1_, 'x2': x2_})
Xnew = design.transform(new_df)
yhat = results.predict(Xnew)
yhat


### Visualize

In [ ]:
# extract the covariates, beta0, beta1, beta2
params = results.params
b0 = params['intercept']
b1 = params['x1']
b2 = params['x2']

# construct a 2D grid for evaluation
# 40 is arbitrary, adjust as needed
x1_grid, x2_grid = np.meshgrid(
    np.linspace(toy_df['x1'].min(), toy_df['x1'].max(), 40),
    np.linspace(toy_df['x2'].min(), toy_df['x2'].max(), 40),
)
yhat_grid = b0 + b1 * x1_grid + b2 * x2_grid

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(toy_df['x1'], toy_df['x2'], toy_df['y'], label='Data')
ax.plot_surface(x1_grid, x2_grid, yhat_grid, alpha=0.5, color='C1', label='OLS Fit')
ax.set_xlabel(r'$X_1$')
ax.set_ylabel(r'$X_2$')
ax.set_zlabel(r'$Y$')
ax.set_title("OLS Fitted Plane")
ax.legend()
ax.view_init(elev=20, azim=15) # change viewpoint
fig.tight_layout() # improves the output


# Fitting with Categorical Features

## Load and Prep Auto Data

In [ ]:
Auto = pd.read_csv("../data/Auto.csv")
Auto.set_index('name', inplace=True)
Auto.cylinders = Auto['cylinders'].astype('category')
Auto.origin = Auto['origin'].astype('category')


In [ ]:
Auto['cylinders'].cat.categories

In [ ]:
fig, ax = plt.subplots()

for cyl, g in Auto.groupby('cylinders'):
    ax.scatter(g['weight'], g['mpg'], alpha=0.7, label=str(cyl))

ax.set_xlabel('Weight')
ax.set_ylabel('MPG')
ax.legend(title='Cylinders')

fig.tight_layout()



## Fitting Against Cylinders

In [ ]:
desgin = MS(['cylinders'])
X = desgin.fit_transform(Auto)
X.head()

In [ ]:
Auto['cylinders'].head()

In [ ]:
model = sm.OLS(Auto['mpg'], X)
results = model.fit()
results.summary()

**NOTE** This does not explicitly represent the 3 cylinder case.

## Fitting Against Cylinders and Weight

In [ ]:
desgin = MS(['cylinders', 'weight', ('cylinders', 'weight')])
X = desgin.fit_transform(Auto)
X.head()

In [ ]:
model = sm.OLS(Auto['mpg'], X)
results = model.fit()
results.summary()

In [ ]:
fig, ax = plt.subplots()

for i, (cyl, g) in enumerate(Auto.groupby('cylinders')):
    ax.scatter(g['weight'], g['mpg'], alpha=0.5, label=str(cyl), color=f'C{i}')
    if cyl>3:
        xrange = np.asarray([g['weight'].min(), g['weight'].max()])
        slope = results.params['weight'] + results.params[f'cylinders[{cyl}]:weight']
        ax.plot(xrange,  results.params['intercept'] + results.params[f'cylinders[{cyl}]'] + slope *xrange, color=f'C{i}')
    else:
        xrange = np.asarray([g['weight'].min(), g['weight'].max()])
        slope = results.params['weight']
        ax.plot(xrange,  results.params['intercept']  + slope *xrange, color=f'C{i}')

ax.set_xlabel('Weight')
ax.set_ylabel('MPG')
ax.legend(title='Cylinders')

fig.tight_layout()


# Outliers and Leverage
Outliers are points that in some way or another appear unusual; they deviate from the trend and typically have high residuals.  Leverage is a way of measuring how significant any point is in the regression.  Outliers which have high residuals and low leverage can safely be dropped, but those with high leverage suggest more data is needed.

## Construct Toy Data

In [ ]:
n = 100
rng = np.random.default_rng(1234)

x_ = np.sort(rng.uniform(-2, 3, n-1))
x_ = np.append(x_, 6)  # add a high leverage point
y_ = 1 + 1.5 * x_ + .2* rng.normal(size=x_.shape)
y_[int(n/2)]+=-5  # add an outlier

y_[-1]+=+2  # add an outlier with high leverage

outlier_df = pd.DataFrame({'x': x_, 'y': y_})
outlier_df.head()
fig, ax = plt.subplots()
outlier_df.plot.scatter('x', 'y', ax=ax,label='Data')
plt.scatter([x_[int(n/2)], x_[-1]], [y_[int(n/2)], y_[-1]],marker='x', color='red', label='Outliers')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.legend()

In [ ]:
design = MS(['x'])
X = design.fit_transform(outlier_df)
model = sm.OLS(outlier_df['y'], X)
results = model.fit()
summarize(results)

In [ ]:
infl = results.get_influence()
infl.hat_matrix_diag

Plot the leverage, which measures the influence of a given point on the regression.  Higher leverage, more influence.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(np.arange(X.shape[0]), infl.hat_matrix_diag)
ax.set_xlabel('Index')
ax.set_ylabel('Leverage')
print('Index of max leverage point:', np.argmax(infl.hat_matrix_diag))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(infl.hat_matrix_diag, infl.resid_studentized_internal)
ax.set_xlabel('Leverage')
ax.set_ylabel('Studentized Residuals')

idx_max_lev = np.argmax(np.abs(infl.hat_matrix_diag))
idx_max_resid = np.argmax(np.abs(infl.resid_studentized_internal))

print('Index of max leverage point:',idx_max_lev)
print('Index of max Studentized Residuals:', idx_max_resid)

In [ ]:
b0 = results.params['intercept']
b1 = results.params['x']

fig, ax = plt.subplots()
outlier_df.plot.scatter('x', 'y', ax=ax,label='Data')
ax.scatter([x_[idx_max_lev]], [y_[idx_max_lev]],marker='x', color='red', label='Max Leverage')
ax.scatter([x_[idx_max_resid]], [y_[idx_max_resid]],marker='+', color='green', label='Max Studentized Residual')
ax.plot(x_, b0 + b1 * x_, color='orange', label='Fitted Line')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.legend()

Points with low leverage and high Studentized residuals are likely outliers.  Dangerous points have high residuals AND high leverage.

Compare with and without high levefrage point

In [ ]:
design = MS(['x'])
X = design.fit_transform(outlier_df[:-1])  # remove high leverage outlier
model2 = sm.OLS(outlier_df['y'][:-1], X)
results2 = model2.fit()
summarize(results2)

In [ ]:
b0 = results.params['intercept']
b1 = results.params['x']
c0 = results2.params['intercept']
c1 = results2.params['x']

fig, ax = plt.subplots()
outlier_df.plot.scatter('x', 'y', ax=ax,label='Data')
ax.scatter([x_[idx_max_lev]], [y_[idx_max_lev]],marker='x', color='red', label='Max Leverage')
ax.scatter([x_[idx_max_resid]], [y_[idx_max_resid]],marker='+', color='green', label='Max Studentized Residual')
ax.plot(x_, b0 + b1 * x_, color='orange', label='Fitted Line')
ax.plot(x_, c0 + c1 * x_, color='purple', label='Fitted Line without High Leverage')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.legend()